# Kronos cloud scan (free GPU)

Run on **Kaggle** or **Google Colab** with a GPU runtime (Runtime / Settings -> GPU), then Run all.
Self-contained: clones the public Kronos repo, no private code needed. ~minutes on a free T4.

Diversified universe (bonds/gold/international/sector ETFs/laggards) on purpose, to test whether the
short-bias seen in the uptrend-only universe is real regime-reading or just a bearish lean on everything.

In [ ]:
!pip -q install safetensors huggingface_hub einops yfinance >/dev/null 2>&1
!git clone -q https://github.com/shiyu-coder/Kronos /kaggle/working/kronos 2>/dev/null || git clone -q https://github.com/shiyu-coder/Kronos ./kronos
import sys, os
sys.path.insert(0, '/kaggle/working/kronos' if os.path.isdir('/kaggle/working/kronos') else './kronos')
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
import numpy as np, pandas as pd, time, torch
from model import Kronos, KronosTokenizer, KronosPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = KronosTokenizer.from_pretrained('NeoQuasar/Kronos-Tokenizer-base')
mdl = Kronos.from_pretrained('NeoQuasar/Kronos-small')
predictor = KronosPredictor(mdl, tok, max_context=512, device=device)
print('loaded on', device)

In [ ]:
def forecast_paths(df, pred_len, n_paths, T=1.0, top_p=0.9, batch=16):
    ts = pd.to_datetime(df.index)
    x_df = df[['open','high','low','close','volume']].reset_index(drop=True)
    x_ts = pd.Series(ts)
    y_ts = pd.Series(pd.bdate_range(ts[-1] + pd.Timedelta(days=1), periods=pred_len))
    paths = []; rem = n_paths
    while rem > 0:
        c = min(batch, rem)
        preds = predictor.predict_batch(df_list=[x_df]*c, x_timestamp_list=[x_ts]*c,
                                        y_timestamp_list=[y_ts]*c, pred_len=pred_len,
                                        T=T, top_p=top_p, sample_count=1, verbose=False)
        for p in preds:
            paths.append((p['close'].to_numpy(), p['high'].to_numpy(), p['low'].to_numpy()))
        rem -= c
    last_close = float(x_df['close'].iloc[-1])
    finals = np.array([c[-1] for c,_,_ in paths])
    rets = (finals - last_close) / last_close
    return dict(last_close=last_close, paths=paths, p_up=float((finals>last_close).mean()),
                exp_ret=float(rets.mean()*100), sigma=float(rets.std()*100))

def atr(df, period=14):
    h,l,c = df['high'],df['low'],df['close']; pc = c.shift(1)
    tr = pd.concat([(h-l),(h-pc).abs(),(l-pc).abs()], axis=1).max(axis=1)
    return float(tr.tail(period).mean())

def hit_probs(fc, entry, stop, tp, direction):
    risk = abs(entry-stop); tp_first=sl_first=prof_neither=0; rs=[]
    for close, high, low in fc['paths']:
        outcome=None
        for hi, lo in zip(high, low):
            if direction=='long':
                if lo<=stop: outcome='sl'; break
                if hi>=tp: outcome='tp'; break
            else:
                if hi>=stop: outcome='sl'; break
                if lo<=tp: outcome='tp'; break
        if outcome=='tp': tp_first+=1; rs.append(abs(tp-entry)/risk)
        elif outcome=='sl': sl_first+=1; rs.append(-1.0)
        else:
            final=close[-1]; pnl=(final-entry) if direction=='long' else (entry-final)
            rs.append(pnl/risk); prof_neither += pnl>0
    n=len(fc['paths'])
    return (tp_first+prof_neither)/n, float(np.mean(rs))

def plan(sym, df, fc, stop_mult=1.5, rr_min=1.5, rr_max=4.0):
    direction = 'long' if fc['p_up']>=0.5 else 'short'
    p_dir = fc['p_up'] if direction=='long' else 1-fc['p_up']
    a=atr(df); entry=fc['last_close']; rr=rr_min+((p_dir-0.5)/0.5)*(rr_max-rr_min); sd=stop_mult*a
    stop, tp = (entry-sd, entry+rr*sd) if direction=='long' else (entry+sd, entry-rr*sd)
    pp, er = hit_probs(fc, entry, stop, tp, direction)
    return dict(symbol=sym, direction=direction, entry=round(entry,2), stop=round(stop,2),
                tp=round(tp,2), rr=round(rr,2), conviction=round(p_dir,3),
                p_profit=round(pp,3), expected_r=round(er,2), sigma=round(fc['sigma'],2))

In [ ]:
import yfinance as yf

SYMBOLS = ['SPY','QQQ','NVDA','AAPL','MSFT','XOM','JPM','LLY',
           'TLT','GLD','USO','EEM','EFA','XLE','XLF','XLU',
           'F','INTC','PFE','WBA','PARA','T']
PRED_LEN = 10
N_PATHS  = 30
MIN_PROB = 0.60
MIN_ER   = 0.0
GPU_RATE = 0.00   # set to your box's $/hr to see cost (Kaggle/Colab free = 0)

rows=[]; t0=time.time()
for s in SYMBOLS:
    try:
        df = yf.download(s, period='2y', interval='1d', auto_adjust=False, progress=False)
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
        df = df.rename(columns=str.lower)[['open','high','low','close','volume']].dropna()
        rows.append(plan(s, df, forecast_paths(df, PRED_LEN, N_PATHS)))
        print(s, 'done')
    except Exception as e:
        print(s, 'ERR', e)

res = pd.DataFrame(rows)
passed = res[(res.p_profit>=MIN_PROB) & (res.expected_r>=MIN_ER)].sort_values('p_profit', ascending=False)
el = time.time()-t0
print(f"\n{len(passed)}/{len(res)} passed | {int((passed.direction=='long').sum())} long / {int((passed.direction=='short').sum())} short")
print(f"{el:.0f}s on {device} | est cost @ ${GPU_RATE}/hr: ${el/3600*GPU_RATE:.3f}")
passed